In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/daily_pollutants.csv")
df["date"] = pd.to_datetime(df["date"])

# Remove 2025
df = df[df["date"].dt.year < 2025]
print(f"Shape after removing 2025: {df.shape}")

Shape after removing 2025: (35930597, 5)


In [2]:
# Pivot: one row = one station + one day, columns = pollutants
df_pivot = df.pivot_table(
    index=["country", "station", "date"],
    columns="pollutant",
    values="value",
    aggfunc="mean"
).reset_index()

# Flatten column names
df_pivot.columns.name = None

print(f"Shape: {df_pivot.shape}")
print(f"\nMissing values per pollutant:")
print(df_pivot[["CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]].isna().sum())
print(f"\nSample:")
df_pivot.head()

Shape: (35930597, 9)

Missing values per pollutant:
CO       33382941
NO2      26867989
O3       29124658
PM10     27050751
PM2.5    31499260
SO2      31727386
dtype: int64

Sample:


,country,station,date,CO,NO2,O3,PM10,PM2.5,SO2
0,DE,DE/SPO.DE_DEBB007_NO2_dataGroup1,2013-01-01,NaN,9.789583,NaN,NaN,NaN,NaN
1,DE,DE/SPO.DE_DEBB007_NO2_dataGroup1,2013-01-02,NaN,10.552917,NaN,NaN,NaN,NaN
2,DE,DE/SPO.DE_DEBB007_NO2_dataGroup1,2013-01-03,NaN,12.682500,NaN,NaN,NaN,NaN
3,DE,DE/SPO.DE_DEBB007_NO2_dataGroup1,2013-01-04,NaN,15.866667,NaN,NaN,NaN,NaN
4,DE,DE/SPO.DE_DEBB007_NO2_dataGroup1,2013-01-05,NaN,5.945417,NaN,NaN,NaN,NaN


In [3]:
# Look at station ID patterns per country
for c in sorted(df["country"].unique()):
    samples = df[df["country"] == c]["station"].unique()[:5]
    print(f"\n{c}:")
    for s in samples:
        print(f"  {s}")


DE:
  DE/SPO.DE_DEBB032_CO_dataGroup1
  DE/SPO.DE_DEBB045_CO_dataGroup1
  DE/SPO.DE_DEBB054_CO_dataGroup1
  DE/SPO.DE_DEBB086_CO_dataGroup1
  DE/SPO.DE_DEBB087_CO_dataGroup1

ES:
  ES/SP_01022001_6_48
  ES/SP_01036004_6_48
  ES/SP_01055001_6_48
  ES/SP_01059008_6_48
  ES/SP_01059009_6_48

FR:
  FR/SPO-FR01005_10
  FR/SPO-FR01059_10
  FR/SPO-FR01064_10
  FR/SPO-FR02008_10
  FR/SPO-FR02031_10

IT:
  IT/SPO.IT0063A_10_NDIR_2001-10-01_00:00:00
  IT/SPO.IT0187A_10_IR-GFC_1990-04-18_00:00:00
  IT/SPO.IT0459A_10_NDIR_1997-09-02_00:00:00
  IT/SPO.IT0460A_10_NDIR_2006-12-01_00:00:00
  IT/SPO.IT0469A_10_NDIR_1997-01-01_00:00:00

NL:
  NL/SPO-NL00007_00010_100
  NL/SPO-NL00012_00010_100
  NL/SPO-NL00014_00010_100
  NL/SPO-NL00487_00010_100
  NL/SPO-NL00488_00010_100

PL:
  PL/SPO_PL0008A_10_002
  PL/SPO_PL0012A_10_001
  PL/SPO_PL0014A_10_001
  PL/SPO_PL0031A_10_001
  PL/SPO_PL0039A_10_001


In [4]:
# Extract real station code from the messy IDs
def extract_station_id(station):
    """Pull out the physical station code, ignoring pollutant/method info."""
    import re
    
    if station.startswith("DE/"):
        # DE/SPO.DE_DEBB032_CO_dataGroup1 -> DEBB032
        m = re.search(r"DE_(\w+?)_", station)
        return f"DE_{m.group(1)}" if m else station
    
    elif station.startswith("ES/"):
        # ES/SP_01022001_6_48 -> 01022001
        m = re.search(r"SP_(\d+)_", station)
        return f"ES_{m.group(1)}" if m else station
    
    elif station.startswith("FR/"):
        # FR/SPO-FR01005_10 -> FR01005
        m = re.search(r"SPO-(FR\d+)", station)
        return f"FR_{m.group(1)}" if m else station
    
    elif station.startswith("IT/"):
        # IT/SPO.IT0063A_10_NDIR_... -> IT0063A
        m = re.search(r"SPO\.(IT\w+?)_", station)
        return f"IT_{m.group(1)}" if m else station
    
    elif station.startswith("NL/"):
        # NL/SPO-NL00007_00010_100 -> NL00007
        m = re.search(r"SPO-(NL\d+)", station)
        return f"NL_{m.group(1)}" if m else station
    
    elif station.startswith("PL/"):
        # PL/SPO_PL0008A_10_002 -> PL0008A
        m = re.search(r"SPO_(PL\w+?)_", station)
        return f"PL_{m.group(1)}" if m else station
    
    return station


# Test on a few examples
test_cases = [
    "DE/SPO.DE_DEBB032_CO_dataGroup1",
    "ES/SP_01022001_6_48",
    "FR/SPO-FR01005_10",
    "IT/SPO.IT0063A_10_NDIR_2001-10-01_00:00:00",
    "NL/SPO-NL00007_00010_100",
    "PL/SPO_PL0008A_10_002",
]

for t in test_cases:
    print(f"  {t:55s} -> {extract_station_id(t)}")

  DE/SPO.DE_DEBB032_CO_dataGroup1                         -> DE_DEBB032
  ES/SP_01022001_6_48                                     -> ES_01022001
  FR/SPO-FR01005_10                                       -> FR_FR01005
  IT/SPO.IT0063A_10_NDIR_2001-10-01_00:00:00              -> IT_IT0063A
  NL/SPO-NL00007_00010_100                                -> NL_NL00007
  PL/SPO_PL0008A_10_002                                   -> PL_PL0008A


All six countries parsed cleanly. Now we apply it to the entire dataset and pivot again:

In [5]:
# Apply to full dataset
df["station_id"] = df["station"].apply(extract_station_id)

print(f"Original stations: {df['station'].nunique()}")
print(f"Real stations: {df['station_id'].nunique()}")

# Pivot again with clean station IDs
df_pivot = df.pivot_table(
    index=["country", "station_id", "date"],
    columns="pollutant",
    values="value",
    aggfunc="mean"
).reset_index()

df_pivot.columns.name = None

print(f"\nPivot shape: {df_pivot.shape}")
print(f"\nMissing values:")
print(df_pivot[["CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]].isna().sum())
print(f"\nNon-null counts:")
print(df_pivot[["CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]].notna().sum())

Original stations: 12996
Real stations: 3789

Pivot shape: (11653711, 9)

Missing values:
CO       9106418
NO2      2593770
O3       4847772
PM10     3460223
PM2.5    7422327
SO2      7450860
dtype: int64

Non-null counts:
CO       2547293
NO2      9059941
O3       6805939
PM10     8193488
PM2.5    4231384
SO2      4202851
dtype: int64


In [6]:
# Keep rows where at least PM2.5 or PM10 is available
has_pm = df_pivot["PM2.5"].notna() | df_pivot["PM10"].notna()
df_clean = df_pivot[has_pm].copy()

print(f"Before: {len(df_pivot):,}")
print(f"After (has PM2.5 or PM10): {len(df_clean):,}")
print(f"Dropped: {len(df_pivot) - len(df_clean):,}")

print(f"\nCoverage in cleaned data:")
for col in ["CO", "NO2", "O3", "PM10", "PM2.5", "SO2"]:
    pct = df_clean[col].notna().mean() * 100
    print(f"  {col:6s}: {pct:.1f}%")

Before: 11,653,711
After (has PM2.5 or PM10): 8,637,792
Dropped: 3,015,919

Coverage in cleaned data:
  CO    : 24.5%
  NO2   : 84.0%
  O3    : 55.3%
  PM10  : 94.9%
  PM2.5 : 49.0%
  SO2   : 37.7%


In [9]:
# AQI Calculator — EPA breakpoints
AQI_BREAKPOINTS = {
    "PM2.5": [
        (0.0, 12.0, 0, 50), (12.1, 35.4, 51, 100), (35.5, 55.4, 101, 150),
        (55.5, 150.4, 151, 200), (150.5, 250.4, 201, 300), (250.5, 500.4, 301, 500),
    ],
    "PM10": [
        (0, 54, 0, 50), (55, 154, 51, 100), (155, 254, 101, 150),
        (255, 354, 151, 200), (355, 424, 201, 300), (425, 604, 301, 500),
    ],
    "NO2": [
        (0, 53, 0, 50), (54, 100, 51, 100), (101, 360, 101, 150),
        (361, 649, 151, 200), (650, 1249, 201, 300), (1250, 2049, 301, 500),
    ],
    "O3": [
        (0, 54, 0, 50), (55, 70, 51, 100), (71, 85, 101, 150),
        (86, 105, 151, 200), (106, 200, 201, 300),
    ],
    "SO2": [
        (0, 35, 0, 50), (36, 75, 51, 100), (76, 185, 101, 150),
        (186, 304, 151, 200), (305, 604, 201, 300), (605, 1004, 301, 500),
    ],
    "CO": [
        (0.0, 4.4, 0, 50), (4.5, 9.4, 51, 100), (9.5, 12.4, 101, 150),
        (12.5, 15.4, 151, 200), (15.5, 30.4, 201, 300), (30.5, 50.4, 301, 500),
    ],
}

def calc_sub_index(value, pollutant):
    if pd.isna(value) or value < 0:
        return np.nan
    for c_low, c_high, i_low, i_high in AQI_BREAKPOINTS.get(pollutant, []):
        if c_low <= value <= c_high:
            return (i_high - i_low) / (c_high - c_low) * (value - c_low) + i_low
    return 500  # exceeds all breakpoints

# Calculate sub-indices for each pollutant
for poll in ["PM2.5", "PM10", "NO2", "O3", "SO2", "CO"]:
    df_clean[f"aqi_{poll}"] = df_clean[poll].apply(lambda v: calc_sub_index(v, poll))

# AQI = max of all sub-indices
aqi_cols = [f"aqi_{p}" for p in ["PM2.5", "PM10", "NO2", "O3", "SO2", "CO"]]
df_clean["aqi"] = df_clean[aqi_cols].max(axis=1)

print(f"AQI calculated for {df_clean['aqi'].notna().sum():,} rows")
print(f"\nAQI stats:")
print(df_clean["aqi"].describe())

AQI calculated for 8,637,792 rows

AQI stats:
count    8.637792e+06
mean     7.012144e+01
std      8.243190e+01
min      0.000000e+00
25%      2.925314e+01
50%      4.731867e+01
75%      7.912328e+01
max      5.000000e+02
Name: aqi, dtype: float64


In [10]:
# AQI categories
def aqi_category(aqi):
    if pd.isna(aqi): return "Unknown"
    if aqi <= 50: return "Good"
    if aqi <= 100: return "Moderate"
    if aqi <= 150: return "Unhealthy (Sensitive)"
    if aqi <= 200: return "Unhealthy"
    if aqi <= 300: return "Very Unhealthy"
    return "Hazardous"

df_clean["aqi_category"] = df_clean["aqi"].apply(aqi_category)

# Distribution
print("AQI category distribution:")
counts = df_clean["aqi_category"].value_counts()
for cat in ["Good", "Moderate", "Unhealthy (Sensitive)", "Unhealthy", "Very Unhealthy", "Hazardous"]:
    if cat in counts:
        pct = counts[cat] / len(df_clean) * 100
        print(f"  {cat:25s}: {counts[cat]:>10,}  ({pct:.1f}%)")

AQI category distribution:
  Good                     :  4,670,558  (54.1%)
  Moderate                 :  2,498,362  (28.9%)
  Unhealthy (Sensitive)    :    821,875  (9.5%)
  Unhealthy                :    350,398  (4.1%)
  Very Unhealthy           :     63,658  (0.7%)
  Hazardous                :    232,941  (2.7%)


In [ ]:
# Time features
df_clean["year"] = df_clean["date"].dt.year
df_clean["month"] = df_clean["date"].dt.month
df_clean["day_of_week"] = df_clean["date"].dt.dayofweek  # 0=Monday
df_clean["day_of_year"] = df_clean["date"].dt.dayofyear
df_clean["season"] = df_clean["month"].map({
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
})

print(f"Final shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

# Save
df_clean.to_csv("../data/processed/aqi_dataset.csv", index=False)
print(f"\nSaved to data/processed/aqi_dataset.csv")
print(f"File size: {pd.io.common.file_exists('../data/processed/aqi_dataset.csv')}")

Final shape: (8637792, 22)
Columns: ['country', 'station_id', 'date', 'CO', 'NO2', 'O3', 'PM10', 'PM2.5', 'SO2', 'aqi_PM2.5', 'aqi_PM10', 'aqi_NO2', 'aqi_O3', 'aqi_SO2', 'aqi_CO', 'aqi', 'aqi_category', 'year', 'month', 'day_of_week', 'day_of_year', 'season']
